# CRISP-DM Case 2: Startup 50 Profit Prediction

這是一個使用 CRISP-DM 解決多元線性迴歸 (Multiple Linear Regression) 問題的範例。
我們將根據新創公司的研發支出、行政支出、行銷支出與所在地區，預測公司的利潤 (Profit)。

## Step 1: Load Data + Scatter Plot

這一步是先把資料讀進來，並用圖表觀察資料長什麼樣子。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 載入資料 (請確保 50_Startups.csv 已放置在 data/raw 目錄)
# 這裡我們自動下載如果檔案不存在的話
data_path = "../data/raw/50_Startups.csv"
if not os.path.exists(data_path):
    os.makedirs("../data/raw", exist_ok=True)
    url = "https://raw.githubusercontent.com/krishnaik06/Multiple-Linear-Regression/master/50_Startups.csv"
    df = pd.read_csv(url)
    df.to_csv(data_path, index=False)
else:
    df = pd.read_csv(data_path)

df.head()

### 產生圖表

如果散佈圖呈現明顯向上趨勢，代表該特徵可能與 Profit 有正相關。

In [ ]:
# 1. R&D Spend vs Profit
plt.figure(figsize=(15, 4))
plt.subplot(1, 3, 1)
plt.scatter(df["R&D Spend"], df["Profit"])
plt.xlabel("R&D Spend")
plt.ylabel("Profit")
plt.title("R&D Spend vs Profit")

# 2. Administration vs Profit
plt.subplot(1, 3, 2)
plt.scatter(df["Administration"], df["Profit"], color='orange')
plt.xlabel("Administration")
plt.ylabel("Profit")
plt.title("Administration vs Profit")

# 3. Marketing Spend vs Profit
plt.subplot(1, 3, 3)
plt.scatter(df["Marketing Spend"], df["Profit"], color='green')
plt.xlabel("Marketing Spend")
plt.ylabel("Profit")
plt.title("Marketing vs Profit")

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap (只取數值欄位)
numeric_df = df.select_dtypes(include=['float64', 'int64'])
plt.figure(figsize=(6, 5))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## Step 2: Preprocessing

因為 `State` 是類別型特徵，我們需要做 One-Hot Encoding。
另外，Startup 50 不是時間序列，所以我們可以隨機切分 (Train Test Split)。

In [ ]:
from sklearn.model_selection import train_test_split

# 將資料分為特徵 (X) 與目標變數 (y)
X = df.drop("Profit", axis=1)
y = df["Profit"]

# Train Test Split (80% 訓練, 20% 測試)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Train set: {X_train.shape}, Test set: {X_test.shape}")

## Step 3 & Step 5 結合: 建立 Pipeline 與模型

為了避免部署時 One-Hot Encoding 欄位與訓練時不一致，我們使用 Pipeline。
Pipeline 可以把「資料前處理」和「模型」綁在一起。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

# 定義數值型與類別型欄位
numeric_features = ["R&D Spend", "Administration", "Marketing Spend"]
categorical_features = ["State"]

# 建立前處理器
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

# 建立 Pipeline
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression())
    ]
)

# 訓練模型 (傳入包含 State 字串的 DataFrame 即可，Pipeline 會自動做 One-Hot)
model.fit(X_train, y_train)

# 預測
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

### 係數解讀
係數代表當某個特徵增加 1 單位時，在其他條件不變的情況下，Profit 預測值會增加或減少多少。

In [ ]:
# 取得 Linear Regression 模型的係數
lr_model = model.named_steps["regressor"]
cat_encoder = model.named_steps["preprocessor"].named_transformers_["cat"]

# 取得 One-Hot Encoding 產生的欄位名稱
encoded_cat_cols = cat_encoder.get_feature_names_out(categorical_features)
all_feature_names = list(encoded_cat_cols) + numeric_features

# 顯示係數
for name, coef in zip(all_feature_names, lr_model.coef_):
    print(f"{name}: {coef:.4f}")

## Step 4: Evaluation

來看看這個多元線性迴歸模型的表現。

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

mse = mean_squared_error(y_test, y_pred_test)
mae = mean_absolute_error(y_test, y_pred_test)
r2 = r2_score(y_test, y_pred_test)
train_r2 = r2_score(y_train, y_pred_train)

print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")

### Actual vs Predicted Plot

如果點越接近斜對角線，代表預測越準。

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred_test)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel("Actual Profit")
plt.ylabel("Predicted Profit")
plt.title("Actual vs Predicted Profit")
plt.show()

## 部署模型 (Save Model)

把 Pipeline 整個存起來！

In [ ]:
import pickle
import gzip
import os

os.makedirs("../models", exist_ok=True)
model_path = "../models/startup_model.pkl.gz"

with gzip.open(model_path, "wb") as f:
    pickle.dump(model, f)
    
print(f"Pipeline saved to {model_path}")